# 12 - Fetch torch 2.10.0+cu128 for the GPU kernel (CPU, internet, ~5 min)

The wheelhouse's `vllm-0.19.1` declares `torch==2.10.0`. Kaggle's GPU image ships
torch 2.11.0+cu128, so `vllm/_C.abi3.so` will not link
(`undefined symbol: _ZN3c1013MessageLoggerC1EPKciib`, a libtorch C++ symbol).

torch is not in the wheelhouse and the GPU kernel has no internet, so this CPU
kernel downloads it and the GPU kernel consumes this kernel's **output** as a data
source. No GPU quota, no local bandwidth.

Output: `/kaggle/working/torch2100/*.whl`

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

OUT = Path('/kaggle/working/torch2100')
OUT.mkdir(parents=True, exist_ok=True)


def say(msg=''):
    print('[' + time.strftime('%H:%M:%S') + '] ' + str(msg), flush=True)


# --platform/--python-version force a cross-download: this kernel is CPU and may
# even be a different Python, but the wheels have to match the GPU image, which is
# cp313 / manylinux_2_28 / x86_64. Without these flags pip would resolve for THIS
# interpreter and the GPU kernel would reject every wheel.
#
# --extra-index-url is required, not optional: torch's +cu128 build lives on the
# PyTorch index, but its nvidia-* dependencies (nvidia-cuda-nvrtc-cu12==12.8.93 and
# friends) live on PyPI. With only the PyTorch index, pip resolved torch and then
# died with "Could not find a version that satisfies nvidia-cuda-nvrtc-cu12" and
# downloaded nothing at all.
BASE = [sys.executable, '-m', 'pip', 'download',
        '--index-url', 'https://download.pytorch.org/whl/cu128',
        '--extra-index-url', 'https://pypi.org/simple',
        '--only-binary=:all:',
        '--python-version', '313',
        # --platform is EXACT, not a floor: pip accepts only the tags listed, and
        # does not infer that manylinux_2_28 can run a manylinux_2_12 wheel. torch
        # itself is manylinux_2_28, but its nvidia-* deps are tagged all over the
        # ladder -- nvidia_cuda_nvrtc_cu12 12.8.93 is
        # `manylinux2010_x86_64.manylinux_2_12_x86_64`. With only _2_28 listed, pip
        # reported "from versions: none" for a wheel that exists, twice. So the
        # whole ladder is passed; repeated --platform is supported.
        '--platform', 'manylinux_2_28_x86_64',
        '--platform', 'manylinux_2_27_x86_64',
        '--platform', 'manylinux_2_24_x86_64',
        '--platform', 'manylinux_2_17_x86_64',
        '--platform', 'manylinux2014_x86_64',
        '--platform', 'manylinux_2_12_x86_64',
        '--platform', 'manylinux2010_x86_64',
        '--platform', 'manylinux1_x86_64',
        '-d', str(OUT)]

# WITH dependencies. torch 2.10 and 2.11 pin different nvidia-* versions, and the
# image's are installed for 2.11; mixing them is exactly the class of mismatch
# being fixed here, so the matching set is fetched rather than assumed compatible.
#
# The +cu128 local version is REQUIRED in the specifier, not decoration. With both
# indexes visible, `torch==2.10.0` matches the PyPI build AND the PyTorch index's
# 2.10.0+cu128, and pip refuses: "Cannot install torch==2.10.0 and
# torch==2.10.0+cu128 because these package versions have conflicting
# dependencies." Naming the local version disambiguates it.
TARGETS = ['torch==2.10.0+cu128', 'torchvision==0.25.0+cu128',
           'torchaudio==2.10.0+cu128']

say('downloading ' + ', '.join(TARGETS))
r = subprocess.run(BASE + TARGETS, capture_output=True, text=True)
say('pip rc ' + str(r.returncode))
if r.returncode != 0:
    say((r.stdout or '')[-2000:])
    say((r.stderr or '')[-3000:])
    # Retry one at a time: a single unresolvable target otherwise aborts the batch
    # and leaves nothing, which is the same trap as the wheelhouse bulk install.
    say('retrying individually')
    for t in TARGETS:
        rr = subprocess.run(BASE + [t], capture_output=True, text=True)
        say('  ' + t + ' rc ' + str(rr.returncode))
        if rr.returncode != 0:
            say('    ' + (rr.stderr or '')[-800:])

wheels = sorted(OUT.glob('*.whl'))
total = sum(w.stat().st_size for w in wheels)
say('')
say(str(len(wheels)) + ' wheel(s), ' + format(total / 1e9, '.2f') + ' GB')
for w in wheels:
    say('  ' + w.name + '  ' + format(w.stat().st_size / 1e6, '.1f') + ' MB')

# The one wheel the whole exercise is for. Fail loudly if it is absent, because a
# partial download would send the GPU kernel back into the same ABI error.
have_torch = [w for w in wheels if w.name.startswith('torch-2.10.0')]
say('')
say('torch 2.10.0 present: ' + str(bool(have_torch)))
(Path('/kaggle/working') / 'fetch_manifest.json').write_text(
    json.dumps({'torch': '2.10.0', 'wheels': [w.name for w in wheels],
                 'bytes': total}, indent=2), encoding='utf-8')
assert have_torch, 'torch 2.10.0 did not download; the GPU kernel cannot be fixed without it'
say('DONE')